# Core Focus Increase Analysis
Runs Grad-CAM + SAM across fish/marine dataset classes, finds images where contextual factor descriptions most increase core-region attention.

## ⚙️ Configuration — Edit This Cell Between Runs

In [ ]:
# ─── TOGGLE DATASET ──────────────────────────────────────────────────────────
DATASET = 'marine'   # 'fish' or 'marine'

# ─── DATA PATH ───────────────────────────────────────────────────────────────
DATA_ROOT = '/content/dissertation_perceptionCLIP/datasets/data'

# ─── DESCRIPTIONS ────────────────────────────────────────────────────────────
# First entry = baseline (no context), rest = factor variants.
# The INCREASE is measured as: core%(factor) - core%(baseline) for each factor,
# then max increase across all factors is used to rank the image.
TEXTS_TO_ADD = [
    '.',                       # baseline — no context
    ', underwater.',           # factor 1
    ', on the ocean floor.',   # factor 2
    ', near coral reef.',      # factor 3
]

# ─── SAMPLING ────────────────────────────────────────────────────────────────
MAX_IMAGES_PER_CLASS = 100   # cap per class (uses random sample if more exist)
RANDOM_SEED          = 42

# ─── SAM MASK SELECTION ──────────────────────────────────────────────────────
# We show the first N masks and let you pick interactively.
# Set INTERACTIVE_MASK = True to pause and ask for each image.
# Set INTERACTIVE_MASK = False to auto-select mask index AUTO_MASK_IDX.
INTERACTIVE_MASK = True
AUTO_MASK_IDX    = 0   # used when INTERACTIVE_MASK = False
N_MASKS_TO_SHOW  = 5   # how many candidate masks to display

# ─── OUTPUT ──────────────────────────────────────────────────────────────────
TOP_N        = 100    # max results to show (all with positive increase, sorted)
RESULTS_DIR  = './core_focus_results'

# ─── SAM CHECKPOINT ──────────────────────────────────────────────────────────
SAM_CHECKPOINT = 'sam_vit_h_4b8939.pth'
SAM_MODEL_TYPE = 'vit_h'

print(f'Dataset : {DATASET}')
print(f'Texts   : {TEXTS_TO_ADD}')
print(f'Max imgs: {MAX_IMAGES_PER_CLASS}/class')
print(f'Interactive mask selection: {INTERACTIVE_MASK}')


## 1. Install & Import Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install opencv-python matplotlib --quiet
!{sys.executable} -m pip install 'git+https://github.com/facebookresearch/segment-anything.git' --quiet
!{sys.executable} -m pip install git+https://github.com/openai/CLIP.git --quiet

# TorchRay for Grad-CAM
try:
    import torchray
except ImportError:
    !{sys.executable} -m pip install git+https://github.com/facebookresearch/TorchRay.git --quiet

# Download SAM weights if not present
import os
if not os.path.exists(SAM_CHECKPOINT):
    print('Downloading SAM checkpoint (~2.4 GB) ...')
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
    print('Done.')
else:
    print('SAM checkpoint already present.')


In [ ]:
import os, random, glob, json
from pathlib import Path
from PIL import Image
from IPython.display import display, clear_output

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn.functional as F
from torchvision import transforms
from torchvision.utils import save_image

import clip
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
from torchray.attribution.common import Probe, get_module
from torchray.attribution.grad_cam import gradient_to_grad_cam_saliency

# Determinism
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
os.makedirs(RESULTS_DIR, exist_ok=True)


## 2. Class Lists

In [ ]:
FISH_CLASSES = [
    'Bangus', 'Big Head Carp', 'Black Spotted Barb', 'Catfish', 'Climbing Perch',
    'Fourfinger Threadfin', 'Freshwater Eel', 'Glass Perchlet', 'Goby', 'Gold Fish',
    'Gourami', 'Grass Carp', 'Green Spotted Puffer', 'Indian Carp', 'Indo-Pacific Tarpon',
    'Jaguar Gapote', 'Janitor Fish', 'Knifefish', 'Long-Snouted Pipefish', 'Mosquito Fish',
    'Mudfish', 'Mullet', 'Pangasius', 'Perch', 'Scat Fish', 'Silver Barb', 'Silver Carp',
    'Silver Perch', 'Snakehead', 'Tenpounder', 'Tilapia'
]

MARINE_CLASSES = [
    'clam', 'coral', 'crab', 'dolphin', 'eel', 'fish', 'jellyfish', 'lobster',
    'nudibranch', 'octopus', 'otter', 'penguin', 'puffer fish', 'seahorse', 'seal',
    'sea urchin', 'shark', 'shrimp', 'squid', 'starfish', 'stingray', 'turtle', 'whale'
]

CLASS_LIST = FISH_CLASSES if DATASET == 'fish' else MARINE_CLASSES
TYPE_LABEL = 'a type of fish' if DATASET == 'fish' else 'a marine animal'
DATA_DIR   = os.path.join(DATA_ROOT, DATASET)

print(f'Dataset: {DATASET}  ({len(CLASS_LIST)} classes)')
print(f'Data directory: {DATA_DIR}')


## 3. Load Models (SAM + CLIP)

In [ ]:
# ── SAM ──
print('Loading SAM ...')
sam = sam_model_registry[SAM_MODEL_TYPE](checkpoint=SAM_CHECKPOINT)
sam.to(device=DEVICE)
mask_generator = SamAutomaticMaskGenerator(sam)
print('SAM ready.')

# ── CLIP ──
print('Loading CLIP ViT-B/16 ...')
clip_model, clip_preprocess = clip.load('ViT-B/16', device=DEVICE)
print('CLIP ready.')


## 4. Helper Functions

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Grad-CAM helpers (from diss_gradcams)
# ─────────────────────────────────────────────────────────────────

def reshape_transform(tensor, height=14, width=14):
    result = tensor[1:, :, :].reshape(tensor.size(1), height, width, tensor.size(2))
    result = result.transpose(2, 3).transpose(1, 2)
    return result

def resize_saliency(tensor, saliency, size, mode):
    if size is not False:
        if size is True:
            size = tensor.shape[2:]
        elif isinstance(size, (tuple, list)):
            size = size[::-1]
        saliency = F.interpolate(saliency, size, mode=mode, align_corners=False)
    return saliency


def compute_gradcam_heatmap(image_path, category_id, text_to_add, dataset, class_list, type_label):
    """
    Returns a grayscale heatmap (numpy H×W, values 0-255) for the given image
    and text description.
    """
    image_tensor = clip_preprocess(Image.open(image_path)).unsqueeze(0).to(DEVICE)

    saliency_layer = get_module(clip_model, 'visual.transformer.resblocks.11.ln_1')
    probe = Probe(saliency_layer, target='output')

    prompts = [f'a photo of {cls}, {type_label}{text_to_add}' for cls in class_list]
    text_tokens = clip.tokenize(prompts).to(DEVICE)

    img_feat  = clip_model.encode_image(image_tensor)
    txt_feat  = clip_model.encode_text(text_tokens)
    img_feat  = img_feat  / img_feat.norm(dim=-1, keepdim=True)
    txt_feat  = txt_feat  / txt_feat.norm(dim=-1, keepdim=True)
    logits    = clip_model.logit_scale.exp() * img_feat @ txt_feat.t()
    probs     = logits.softmax(dim=-1)

    probs[0, category_id].backward()

    probe_t      = reshape_transform(probe.data[0])
    probe_t.grad = reshape_transform(probe.data[0].grad)
    saliency     = gradient_to_grad_cam_saliency(probe_t)
    saliency     = resize_saliency(image_tensor, saliency, size=True, mode='bilinear')

    heat = saliency.detach().cpu().numpy()[0, 0]
    mx   = heat.max()
    if mx > 0:
        heat = heat / mx
    return (heat * 255).astype(np.uint8)


# ─────────────────────────────────────────────────────────────────
# SAM helpers
# ─────────────────────────────────────────────────────────────────

def get_sam_masks(image_path):
    """Returns sorted list of SAM masks (largest area first)."""
    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    masks   = mask_generator.generate(img_rgb)
    masks   = sorted(masks, key=lambda m: m['area'], reverse=True)
    return masks, img_rgb


def show_masks_for_selection(image_rgb, masks, n=5):
    """Display the image and up to n candidate masks side by side."""
    n = min(n, len(masks))
    fig, axes = plt.subplots(1, n + 1, figsize=(4 * (n + 1), 4))
    axes[0].imshow(image_rgb)
    axes[0].set_title('Original')
    axes[0].axis('off')
    for i in range(n):
        seg = masks[i]['segmentation']  # bool H×W
        overlay = image_rgb.copy().astype(float)
        overlay[seg]  *= 0.5
        overlay[seg]  += np.array([0, 120, 255]) * 0.5  # blue tint on mask
        overlay        = overlay.clip(0, 255).astype(np.uint8)
        axes[i + 1].imshow(overlay)
        pct = seg.sum() / seg.size * 100
        axes[i + 1].set_title(f'Mask {i}  ({pct:.1f}% area)')
        axes[i + 1].axis('off')
    plt.tight_layout()
    plt.show()
    return n


def pick_mask(masks, image_rgb, image_path, interactive, auto_idx, n_show):
    """Show masks and return selected mask index."""
    n_available = len(masks)
    if n_available == 0:
        return None
    if not interactive:
        return min(auto_idx, n_available - 1)

    n_shown = show_masks_for_selection(image_rgb, masks, n=n_show)
    while True:
        try:
            choice = input(
                f'Image: {os.path.basename(image_path)}\n'
                f'Enter mask index (0–{n_shown - 1}), or "s" to skip this image: '
            ).strip()
            if choice.lower() == 's':
                return None
            idx = int(choice)
            if 0 <= idx < n_available:
                return idx
            print(f'Please enter a number between 0 and {n_shown - 1}.')
        except ValueError:
            print('Invalid input.')


# ─────────────────────────────────────────────────────────────────
# Core / Spurious computation
# ─────────────────────────────────────────────────────────────────

def core_pct(heatmap_gray, mask_bool):
    """
    Given a grayscale heatmap (uint8 H×W) and a boolean mask,
    return the fraction of attention inside the mask.
    """
    img = heatmap_gray.astype(float)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    core     = img[mask_bool].mean()
    spurious = img[~mask_bool].mean()
    total    = core + spurious
    if total < 1e-8:
        return 0.0
    return float(core / total)


print('Helper functions defined.')


## 5. Scan Dataset & Collect Image Paths

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}

def find_images_for_class(class_name, data_dir, max_n, seed):
    """
    Look for a subdirectory matching class_name (case-insensitive),
    return up to max_n image paths.
    """
    # Try exact match first, then case-insensitive
    class_dir = os.path.join(data_dir, class_name)
    if not os.path.isdir(class_dir):
        # fuzzy: replace spaces with underscores or hyphens
        for candidate in os.listdir(data_dir):
            if candidate.lower() == class_name.lower().replace(' ', '_') \
            or candidate.lower() == class_name.lower().replace(' ', '-') \
            or candidate.lower() == class_name.lower():
                class_dir = os.path.join(data_dir, candidate)
                break
        else:
            return []  # not found

    all_imgs = [
        p for p in Path(class_dir).rglob('*')
        if p.suffix.lower() in IMAGE_EXTENSIONS
    ]
    rng = random.Random(seed)
    if len(all_imgs) > max_n:
        all_imgs = rng.sample(all_imgs, max_n)
    return [str(p) for p in all_imgs]


# Build job list: [(class_name, category_id, image_path), ...]
jobs = []
for cat_id, cls in enumerate(CLASS_LIST):
    imgs = find_images_for_class(cls, DATA_DIR, MAX_IMAGES_PER_CLASS, RANDOM_SEED)
    for img_path in imgs:
        jobs.append((cls, cat_id, img_path))

print(f'Total images to process: {len(jobs)}')
class_counts = {}
for cls, _, _ in jobs:
    class_counts[cls] = class_counts.get(cls, 0) + 1
for cls, count in sorted(class_counts.items()):
    print(f'  {cls}: {count} images')


## 6. Main Analysis Loop

For each image:
1. Run SAM → display candidate masks → you pick one
2. Run Grad-CAM for each text description
3. Compute core% for baseline and each factor
4. Record max increase across factors


In [ ]:
results = []  # list of dicts
skipped = []

total = len(jobs)
for job_idx, (class_name, cat_id, img_path) in enumerate(jobs):

    print(f'\n═══ [{job_idx + 1}/{total}] {class_name} — {os.path.basename(img_path)} ═══')

    # ── 1. SAM segmentation ──────────────────────────────────────
    try:
        masks, img_rgb = get_sam_masks(img_path)
    except Exception as e:
        print(f'  SAM failed: {e} — skipping.')
        skipped.append(img_path)
        continue

    if not masks:
        print('  No masks found — skipping.')
        skipped.append(img_path)
        continue

    # ── 2. Mask selection ────────────────────────────────────────
    mask_idx = pick_mask(masks, img_rgb, img_path,
                         interactive=INTERACTIVE_MASK,
                         auto_idx=AUTO_MASK_IDX,
                         n_show=N_MASKS_TO_SHOW)
    if mask_idx is None:
        print('  Skipped by user.')
        skipped.append(img_path)
        continue

    core_mask = masks[mask_idx]['segmentation']  # bool H×W
    print(f'  Using mask {mask_idx}  ({core_mask.sum()} px = {core_mask.mean()*100:.1f}% of image)')

    # ── 3. Grad-CAM for each description ─────────────────────────
    core_scores = []
    for desc_idx, text in enumerate(TEXTS_TO_ADD):
        try:
            # Need fresh grad graph each time
            clip_model.zero_grad()
            heatmap = compute_gradcam_heatmap(
                img_path, cat_id, text, DATASET, CLASS_LIST, TYPE_LABEL
            )
            # Resize mask to match heatmap if needed
            if heatmap.shape != core_mask.shape:
                import skimage.transform as skt
                mask_resized = skt.resize(core_mask.astype(float),
                                          heatmap.shape,
                                          order=0) > 0.5
            else:
                mask_resized = core_mask

            score = core_pct(heatmap, mask_resized)
            core_scores.append(score)
            tag = 'baseline' if desc_idx == 0 else f'factor {desc_idx}'
            print(f'  {tag:12s} → core%: {score:.4f}  ({text!r})')
        except Exception as e:
            print(f'  Grad-CAM failed for desc {desc_idx}: {e}')
            core_scores.append(None)

    # ── 4. Compute max increase ───────────────────────────────────
    if core_scores[0] is None:
        print('  Baseline Grad-CAM failed — skipping.')
        skipped.append(img_path)
        continue

    baseline = core_scores[0]
    increases = []
    for i, s in enumerate(core_scores[1:], start=1):
        if s is not None:
            increases.append((TEXTS_TO_ADD[i], s - baseline, s))

    if not increases:
        print('  All factor Grad-CAMs failed — skipping.')
        skipped.append(img_path)
        continue

    best_text, best_increase, best_core = max(increases, key=lambda x: x[1])
    print(f'  → Best increase: {best_increase:+.4f}  (text={best_text!r}, core%={best_core:.4f})')

    results.append({
        'class_name':    class_name,
        'category_id':   cat_id,
        'image_path':    img_path,
        'mask_idx':      mask_idx,
        'baseline_core': baseline,
        'best_text':     best_text,
        'best_increase': best_increase,
        'best_core':     best_core,
        'all_scores':    {TEXTS_TO_ADD[i]: core_scores[i] for i in range(len(TEXTS_TO_ADD))},
    })

print(f'\n✓ Done. {len(results)} images scored, {len(skipped)} skipped.')


## 7. Ranked Results — Top Images by Core Focus Increase

In [ ]:
# Filter to only positive increases, sort descending
positive = [r for r in results if r['best_increase'] > 0]
positive.sort(key=lambda r: r['best_increase'], reverse=True)

top = positive[:TOP_N]

print(f'Images with positive core focus increase: {len(positive)} / {len(results)}')
print(f'Showing top {len(top)}\n')
print(f'{"Rank":<5} {"Class":<18} {"Baseline":>9} {"Best core":>10} {"Increase":>10}  Best text')
print('─' * 80)
for rank, r in enumerate(top, 1):
    print(f'{rank:<5} {r["class_name"]:<18} {r["baseline_core"]:>9.4f} '
          f'{r["best_core"]:>10.4f} {r["best_increase"]:>+10.4f}  {r["best_text"]!r}')

# Save JSON
out_json = os.path.join(RESULTS_DIR, f'{DATASET}_core_focus_results.json')
with open(out_json, 'w') as f:
    json.dump(top, f, indent=2)
print(f'\nResults saved to {out_json}')


## 8. Visualise Top Results

For each top-ranked image, show: original | SAM mask | baseline Grad-CAM | best-factor Grad-CAM

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm

VIS_TOP_N = min(20, len(top))  # visualise at most 20 to keep runtime reasonable

for rank, r in enumerate(top[:VIS_TOP_N], 1):
    img_path  = r['image_path']
    cls       = r['class_name']
    cat_id    = r['category_id']
    mask_idx  = r['mask_idx']
    best_text = r['best_text']
    increase  = r['best_increase']

    # Load image
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Re-run SAM (or reuse if you stored masks — here we re-run for simplicity)
    masks_vis, _ = get_sam_masks(img_path)
    core_mask    = masks_vis[mask_idx]['segmentation']

    # Baseline heatmap
    clip_model.zero_grad()
    heat_base = compute_gradcam_heatmap(img_path, cat_id, TEXTS_TO_ADD[0],
                                         DATASET, CLASS_LIST, TYPE_LABEL)
    # Best-factor heatmap
    clip_model.zero_grad()
    heat_best = compute_gradcam_heatmap(img_path, cat_id, best_text,
                                         DATASET, CLASS_LIST, TYPE_LABEL)

    # Build mask overlay
    mask_overlay = img_rgb.copy().astype(float)
    mask_overlay[core_mask]  *= 0.5
    mask_overlay[core_mask]  += np.array([0, 120, 255]) * 0.5
    mask_overlay              = mask_overlay.clip(0, 255).astype(np.uint8)

    # Heatmap colourised
    def colourise(heat):
        return cv2.applyColorMap(heat, cv2.COLORMAP_JET)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(
        f'#{rank}  {cls}  |  increase: {increase:+.4f}  |  {os.path.basename(img_path)}',
        fontsize=12, fontweight='bold'
    )

    axes[0].imshow(img_rgb)
    axes[0].set_title('Original')
    axes[0].axis('off')

    axes[1].imshow(mask_overlay)
    axes[1].set_title(f'SAM mask {mask_idx}')
    axes[1].axis('off')

    axes[2].imshow(cv2.cvtColor(colourise(heat_base), cv2.COLOR_BGR2RGB))
    axes[2].set_title(f'Baseline Grad-CAM\ncore%={r["baseline_core"]:.3f}\n{TEXTS_TO_ADD[0]!r}')
    axes[2].axis('off')

    axes[3].imshow(cv2.cvtColor(colourise(heat_best), cv2.COLOR_BGR2RGB))
    axes[3].set_title(f'Best-factor Grad-CAM\ncore%={r["best_core"]:.3f}\n{best_text!r}')
    axes[3].axis('off')

    plt.tight_layout()
    # Save figure
    fig_path = os.path.join(RESULTS_DIR, f'rank_{rank:03d}_{cls}_{os.path.basename(img_path)}.png')
    plt.savefig(fig_path, dpi=100, bbox_inches='tight')
    plt.show()
    print(f'  Saved: {fig_path}')

print(f'\nVisualised top {VIS_TOP_N} results.')


## 9. Class-Level Summary

In [ ]:
from collections import defaultdict

class_stats = defaultdict(list)
for r in positive:
    class_stats[r['class_name']].append(r['best_increase'])

print(f'{'Class':<20} {'Count':>6} {'Mean Δcore%':>12} {'Max Δcore%':>11}')
print('─' * 55)
rows = []
for cls, vals in class_stats.items():
    rows.append((cls, len(vals), np.mean(vals), max(vals)))
rows.sort(key=lambda x: x[2], reverse=True)
for cls, cnt, mean, mx in rows:
    print(f'{cls:<20} {cnt:>6} {mean:>+12.4f} {mx:>+11.4f}')
